<a href="https://colab.research.google.com/github/Rohim110606/BigData26_A_2411533015_Abdur-Rohim/blob/main/Praktikum2/BD_A_L02_2411533015_ABDUR_ROHIM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris
transaksi_mentah.csv dan transaksi_bersih.csv dengan hasil SEED = 42. Apakah
jumlahnya sama? Jelaskan mengapa.

In [1]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [3]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional
    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))


Jumlah baris: 515


In [4]:
print (df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [7]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [9]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [11]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


2.Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk
memeriksa apakah ada harga tidak valid.

In [12]:
# Membuat kolom is_valid_price bernilai True jika price > 0
df['is_valid_price'] = df['price'] > 0

# Memeriksa apakah ada harga yang tidak valid (False atau NaN)
invalid_price_count = (~df['is_valid_price']).sum()
print("Jumlah transaksi dengan harga tidak valid:", invalid_price_count)

# Menampilkan baris yang harganya tidak valid (jika ada)
if invalid_price_count > 0:
    print(df[~df['is_valid_price']])
else:
    print("Semua harga dalam dataset bernilai valid (lebih besar dari 0).")

Jumlah transaksi dengan harga tidak valid: 0
Semua harga dalam dataset bernilai valid (lebih besar dari 0).


3.Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang sudah bersih

In [13]:
# Hitung jumlah transaksi per kategori pada dataset yang sudah bersih
kategori_counts = df['category'].value_counts()
print("Jumlah transaksi per kategori produk:")
print(kategori_counts)

Jumlah transaksi per kategori produk:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


In [14]:
# Upload file CSV ke Google Drive

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Buat folder khusus (jika belum ada)
folder_path = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_path, exist_ok=True)

# Copy file ke folder tersebut
!cp transaksi_mentah.csv "{folder_path}/"
!cp transaksi_bersih.csv "{folder_path}/"

print("✅ File berhasil diupload ke Google Drive!")
print(f"📁 Lokasi: MyDrive/BigData/Praktikum2")
print(f"📄 File yang diupload:")
print(f"   - transaksi_mentah.csv")
print(f"   - transaksi_bersih.csv")

Mounted at /content/drive
✅ File berhasil diupload ke Google Drive!
📁 Lokasi: MyDrive/BigData/Praktikum2
📄 File yang diupload:
   - transaksi_mentah.csv
   - transaksi_bersih.csv
